In [3]:
import json
import requests

query = "capybara"


def search_wikipedia(query:str)->list[dict]:
    #create a functin that returns a wikipedia search
    response=requests.get(
        "https://en.wikipedia.org/w/rest.php/v1/search/page",
        params={"q": query, "limit": 5},
        headers={"User-Agent": "ai-engineering-buildcamp-homework/1.0 (your-email@example.com)"},
    )
    response.raise_for_status()
    return response.json()["pages"]





In [4]:
result=search_wikipedia(query=query)
print (len(result))
print (json.dumps(result))

print(type(result))
print(type(result[0]))
print(result[0])

5
[{"id": 6776, "key": "Capybara", "title": "Capybara", "excerpt": "The <span class=\"searchmatch\">capybara</span> (/k\u00e6p\u026a\u02c8b\u0251\u02d0r\u0259/ kap-ih-BAR-uh) or greater <span class=\"searchmatch\">capybara</span> (Hydrochoerus hydrochaeris) is the largest living rodent, native to all countries in South", "matched_title": null, "anchor": null, "description": "Largest species of rodents", "thumbnail": {"mimetype": "image/jpeg", "width": 60, "height": 40, "duration": null, "url": "//thumb.wikimedia.org/wikipedia/commons/thumb/7/71/Capybaracropped.jpg/60px-Capybaracropped.jpg?utm_source=en.wikipedia.org&utm_campaign=rest&utm_content=thumbnail"}}, {"id": 23188846, "key": "Lesser_capybara", "title": "Lesser capybara", "excerpt": "The lesser <span class=\"searchmatch\">capybara</span> (Hydrochoerus isthmius) is a large semi-aquatic rodent found in South America that has vast similarities with, yet subtle differences", "matched_title": null, "anchor": null, "description": "Spe

In [5]:
count = sum(1 for page in result if "capybara" in page["title"].lower())
print(count)


5


In [6]:
def get_page(pagetitle:str)->str:
    response= requests.get(
        f"https://en.wikipedia.org/w/rest.php/v1/page/{pagetitle}",
        headers={"User-Agent": "ai-engineering-buildcamp-homework/1.0 (your-email@example.com)"},
    )
    response.raise_for_status()
    return response.json()["source"]



In [7]:
result= get_page("Capybara")
print ( len(result))


44767


In [11]:
#define tools and use agentic

from pydantic_ai import Agent
from wikipediaTools import WikipediaTools

wiki_search=WikipediaTools()
tools=[]
tools.append(wiki_search.search_wikipedia)
tools.append(wiki_search.get_page)

agent = Agent(
    "openai:gpt-5.4-nano",
    instructions="You're a documentation assistant. Use the Wikipedia tools to answer questions.",
    tools=tools,
)



In [9]:
def print_messages(messages):
    for m in messages:
        print(m.kind)
        for p in m.parts:
            part_kind = p.part_kind
            if part_kind == 'user-prompt':
                print('USER:', p.content)
            if part_kind == 'tool-call':
                print('TOOL CALL:', p.tool_name, p.args)
            if part_kind == 'tool-return':
                print('TOOL RETURN:', p.tool_name)
            if part_kind == 'text':
                print(p.content)
        print()

In [13]:
#https://en.wikipedia.org/wiki/Capybara
user_prompt =" what is this page about? https://en.wikipedia.org/wiki/Capybara"
result= await agent.run(user_prompt=user_prompt)

print_messages(result.all_messages())


request
USER:  what is this page about? https://en.wikipedia.org/wiki/Capybara

response
TOOL CALL: get_page {"pagetitle":"Capybara"}

request
TOOL RETURN: get_page

response
The Wikipedia page on **capybaras** is about the **greater capybara (*Hydrochoerus hydrochaeris*)**—the **largest living rodent**.

It covers, in particular:
- **Where they live**: native to **most of South America** (all countries except **Chile**), generally near **freshwater**.
- **What they’re like**: physical description (large, barrel-bodied, semi-aquatic), and basics like size/weight and distinctive traits (e.g., webbed feet, near-top-of-head eyes/ears).
- **Diet and behavior**: they’re **herbivores**, often **social** (usually in groups), and are strong swimmers that spend time in water to regulate temperature.
- **Reproduction and communication**: mating habits (often in water), gestation period, group/social organization, and vocalizations.
- **Ecology and predators**: roles in their habitats, lifespan/p

In [14]:
query= "What are the main threats to capybara populations?"
result=await agent.run(user_prompt=query)
print_messages(result.all_messages())

request
USER: What are the main threats to capybara populations?

response
TOOL CALL: search_wikipedia {"query":"capybara main threats to populations habitat loss hunting predation"}

request
TOOL RETURN: search_wikipedia

response
TOOL CALL: search_wikipedia {"query":"capybara threats habitat loss hunting predators"}

request
TOOL RETURN: search_wikipedia

response
TOOL CALL: get_page {"pagetitle":"Capybara"}

request
TOOL RETURN: get_page

response
The main threats to capybara populations are:

- **Hunting by people**: They are hunted for **meat and hide** (and fat/grease in some places). In some areas, this has reduced local numbers.  
- **Killing due to conflict with livestock**: Humans may kill capybaras because they **graze in areas used by cattle**, being viewed as competitors.
- **Predation** (a natural but population-limiting pressure): In the wild, capybaras are preyed on by **large cats** such as **jaguars, ocelots, and cougars**, and also by some **reptiles and birds** (e.g